# Excel to CSV Converter and Cleaner
- Converts all `.xlsx` files in a folder to `.csv`.
- Deletes the original `.xlsx` files after conversion.
# Python Script 

In [1]:
import os
import glob
import pandas as pd

# Define the folder path
folder_path = r"D:\MFC ASSIGNMENT\LinkedIn Data Public"

# Find all .xlsx files in the folder
xlsx_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

# Track converted and deleted files
converted_files = []
deleted_files = []

# Process each .xlsx file
for file_path in xlsx_files:
    file_name = os.path.basename(file_path)
    base_name = os.path.splitext(file_name)[0]
    csv_file_name = base_name + ".csv"
    csv_file_path = os.path.join(folder_path, csv_file_name)

    try:
        # Read and convert to CSV
        df = pd.read_excel(file_path)
        df.to_csv(csv_file_path, index=False)
        converted_files.append(csv_file_name)

        # If conversion succeeded, delete the .xlsx file
        if os.path.exists(csv_file_path):
            os.remove(file_path)
            deleted_files.append(file_name)

    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")

# Output results
print("✅ Converted the following .xlsx files to .csv:")
for name in converted_files:
    print(name)
print(f"\nTotal files converted: {len(converted_files)}")

print("\n✅ Deleted the following original .xlsx files:")
for name in deleted_files:
    print(name)
print(f"\nTotal .xlsx files deleted: {len(deleted_files)}")


✅ Converted the following .xlsx files to .csv:

Total files converted: 0

✅ Deleted the following original .xlsx files:

Total .xlsx files deleted: 0


# CSV File Renamer for LinkedIn Data


In [ ]:
import os
import glob

def get_unique_filename(folder_path, base_name, ext):
    """Return a unique filename in the folder if not exists, else None."""
    final_path = os.path.join(folder_path, f"{base_name}{ext}")
    if os.path.exists(final_path):
        return None  # File with same name exists, skip
    return final_path

# Your folder path
folder_path = r"D:\MFC ASSIGNMENT\Linkedin Data Public"

# Store already used names and renamed files
used_names = set()
renamed_files = []

# Fetch all CSV files from the folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

for file_path in csv_files:
    try:
        file_name = os.path.basename(file_path)
        filename_no_ext = os.path.splitext(file_name)[0]
        ext = ".csv"
        base_name = ""

        # Case 1: File name contains " - "
        if " - " in file_name:
            left, right_with_ext = file_name.split(" - ", 1)
            left_clean = left.replace("_", " ").strip().title()
            right = os.path.splitext(right_with_ext)[0].strip()
            right_clean = right.title()

            if "NO-LASTNAME" in right.upper():
                base_name = left_clean
            elif len(right_clean.split()) < len(left_clean.split()):
                base_name = left_clean
            elif left_clean != right_clean and right_clean in used_names:
                base_name = f"{left_clean} ({right_clean})"
            else:
                base_name = right_clean

        # Case 2: File name has "_" but no " - "
        elif "_" in filename_no_ext:
            base_name = filename_no_ext.replace("_", " ").strip().title()

        # Case 3: Clean name, no "_" or " - "
        else:
            base_name = filename_no_ext.strip().title()

        # Final check for duplicates
        new_file_path = get_unique_filename(folder_path, base_name, ext)

        if new_file_path is None:
            print(f"⚠️ Duplicate file found for '{file_name}', skipping/removing.")
            os.remove(file_path)
            continue

        # Perform rename
        new_file_name = os.path.basename(new_file_path)
        os.rename(file_path, new_file_path)
        used_names.add(base_name)
        renamed_files.append((file_name, new_file_name))

    except Exception as e:
        print(f"❌ Error processing '{file_name}': {e}")

# Summary of renaming
print("\n✅ Renamed CSV Files:")
for old_name, new_name in renamed_files:
    print(f"  {old_name} → {new_name}")

print(f"\n📄 Total files renamed: {len(renamed_files)}")


# CSV Emoji/Encoding Corruption Checker

This script scans all `.csv` files in the `Cleaned_LinkedIn_CSVs` folder to detect **partially corrupted or emoji-tainted cell values** caused by bad encoding or unreadable characters.



In [9]:
import os
import glob
import pandas as pd

# Folder path
folder_path = r"Cleaned_LinkedIn_CSVs"

# Get all CSVs
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Known corrupt emoji/encoding fragments (may expand)
corrupt_fragments = ["ðŸ", "�", "ï", "â", "€", "™", "œ"]

# Prepare error list
corrupt_tail_issues = []

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    try:
        # Try reading in UTF-8, fallback to ISO
        try:
            df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='ISO-8859-1', on_bad_lines='skip')

        # Remove fully empty rows
        df.dropna(how='all', inplace=True)

        for row_idx, row in df.iterrows():
            for col in df.columns:
                val = row[col]

                if pd.isna(val):
                    continue

                val_str = str(val).strip()

                # Skip if the whole value is just corrupted
                if any(val_str.startswith(corrupt) and len(val_str) < 10 for corrupt in corrupt_fragments):
                    continue

                # Check if valid text contains suspicious corrupt tail
                for frag in corrupt_fragments:
                    if frag in val_str and not val_str.startswith(frag):
                        corrupt_tail_issues.append({
                            "File Name": file_name,
                            "Row Number": row_idx + 2,  # +2 for 1-based index and header
                            "Column": col,
                            "Cell Value": val_str,
                            "Error": f"Contains corrupted/emoji fragment: '{frag}'"
                        })
                        break  # Report once per cell

    except Exception as e:
        corrupt_tail_issues.append({
            "File Name": file_name,
            "Row Number": "N/A",
            "Column": "N/A",
            "Cell Value": "N/A",
            "Error": f"Error processing file: {e}"
        })

# Print results
if corrupt_tail_issues:
    print("\n🔍 Partial Corruption or Emoji-Tail Issues Found:\n")
    for issue in corrupt_tail_issues:
        print(
            f"File: {issue['File Name']}\n"
            f"Row: {issue['Row Number']}\n"
            f"Column: {issue['Column']}\n"
            f"Cell: {issue['Cell Value']}\n"
            f"Error: {issue['Error']}\n"
            f"{'-'*40}"
        )
else:
    print("✅ No partial emoji/corrupted fragments found in data.")



🔍 Partial Corruption or Emoji-Tail Issues Found:

File: Alok Raj.csv
Row: 1482
Column: Company
Cell: Insta Dot Analytics™
Error: Contains corrupted/emoji fragment: '™'
----------------------------------------
File: Alok Raj.csv
Row: 1790
Column: Company
Cell: ZIAVII™
Error: Contains corrupted/emoji fragment: '™'
----------------------------------------
File: Alok Raj.csv
Row: 1914
Column: Company
Cell: BIZBoost™ Inc. 🚀
Error: Contains corrupted/emoji fragment: '™'
----------------------------------------
File: Aman Singh.csv
Row: 2467
Column: Company
Cell: Insta Dot Analytics™
Error: Contains corrupted/emoji fragment: '™'
----------------------------------------
File: Anuradha Tiwari.csv
Row: 265
Column: Company
Cell: Insta Dot Analyticsâ?¢
Error: Contains corrupted/emoji fragment: 'â'
----------------------------------------
File: Anuradha Tiwari.csv
Row: 283
Column: Company
Cell: ZIAVIIâ?¢
Error: Contains corrupted/emoji fragment: 'â'
----------------------------------------
File: A

# LinkedIn Data Processor: ZIP Extractor + Cleaner 



In [ ]:
import os
import zipfile
import pandas as pd
import json

def extract_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Extracted to: {extract_to}")

def create_cleaned_csvs_and_adjacency_dict(input_folder, output_folder, json_output_path):
    os.makedirs(output_folder, exist_ok=True)
    adjacency_dict = {}

    for filename in os.listdir(input_folder):
        if filename.endswith('.csv'):
            file_path = os.path.join(input_folder, filename)

            try:
                # Try reading the CSV file
                try:
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')
                except UnicodeDecodeError:
                    df = pd.read_csv(file_path, encoding='ISO-8859-1', on_bad_lines='skip')

                # Get name after dash in filename
# Get name after dash in filename
                if '-' in filename:
                    new_name = filename.split('-')[1].strip()
                else:
                    new_name = os.path.splitext(filename)[0].strip()

                # Strip existing extension just in case
                new_name = os.path.splitext(new_name)[0]
                cleaned_filename = f"{new_name}.csv"

                cleaned_path = os.path.join(output_folder, cleaned_filename)

                # Extract connections
                if 'First Name' in df.columns and 'Last Name' in df.columns:
                    connections = df[['First Name', 'Last Name']].fillna('').apply(
                        lambda row: f"{row['First Name'].strip()} {row['Last Name'].strip()}".strip(), axis=1
                    ).tolist()
                else:
                    connections = []
                    for col in df.columns:
                        if col.lower() in ['connections', 'connected to', 'connected_to', 'name', 'full name']:
                            connections = df[col].dropna().astype(str).str.strip().tolist()
                            break
                    if not connections:
                        print(f"Warning: No name columns found in {filename}. Skipping.")
                        continue

                # Save cleaned CSV
                df.to_csv(cleaned_path, index=False)

                # Add to adjacency dict
                name_key = os.path.splitext(cleaned_filename)[0]
                adjacency_dict[name_key] = connections

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Save adjacency dictionary to JSON
    with open(json_output_path, 'w') as f:
        json.dump(adjacency_dict, f, indent=4)

    print(f"\n✅ Cleaned CSVs saved in: {output_folder}")

# Example usage:
zip_file_path = "LinkedIn Data Public.zip"
extract_to_folder = "unzipped_data"
data_folder = os.path.join(extract_to_folder, "LinkedIn Data Public")
output_cleaned_folder = "Cleaned_LinkedIn_CSVs"

# Run the process
extract_zip(zip_file_path, extract_to_folder)
create_cleaned_csvs_and_adjacency_dict(data_folder, output_cleaned_folder)


Extracted to: unzipped_data

✅ Cleaned CSVs saved in: Cleaned_LinkedIn_CSVs
